# ORIE 5355 Project 2026: Part 1 capacity-constrained pricing


Run `model_training/purchase_model_baseline.ipynb` first; it saves the model loaded by `agents/agent_part1.py`. Our  naive implemented agent chooses prices using predicted purchase probabilities, with a simple inventory-conservation heuristic -- you should replace it with your better strategies.

Inventory is a uniformly random integer from 7 to 20 every 20 customers.

Use `run_project_part1_static_2026.ipynb` for unconstrained static pricing, and `run_project_part2_2026.ipynb` for Part 2 competition. Complete both Part 1 tasks.


In [ ]:
from pathlib import Path
import os
import sys

# Resolve paths once so every notebook uses the project folder.
project_folder = next((
    folder for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / 'project_code_2026_publicshare',
                   parent / 'project_2026/project_code_2026_publicshare',
                   parent / 'private_github/project_2026/project_code_2026_publicshare')
    if (folder / 'pricing_helpers.py').is_file()
), None)
if project_folder is None:
    raise FileNotFoundError('Open the project folder as your notebook workspace.')
os.chdir(project_folder)
sys.path.insert(0, str(project_folder))

import copy
import numpy as np 
import matplotlib.pyplot as plt
import time
from IPython import display

In [ ]:
import make_env_2026 as make_env

### Part 1: capacity extension

In [ ]:
# In part 1, each team's agent will be evaluated alone.
project_part = 1
agentnames = ['agent_part1']

In [ ]:
env, agents = make_env.make_env_agents(
    agentnames = agentnames, 
    project_part = project_part, 
    first_file = 'data/datafile1_2026.csv', 
    second_file='data/datafile2_2026.csv'
    )

# you can replace the agentnames to match whatever agentfiles you create. 
# Do not rename agents/agent_part1.py or the agents/helper_files_here/ folder.

In [ ]:
T = 500 #2500
verbose = False

In [ ]:
env.reset()
customer_covariates, sale, profits, inventories, time_until_replenish = env.get_current_state_customer_to_send_agents()
last_customer_covariates = customer_covariates
inventory_history = []

fig, ax = plt.subplots(figsize=(20, 10))
for t in range(0, T):
    actions = [agent.action(copy.deepcopy((customer_covariates, sale, profits, inventories, time_until_replenish))) for agent in agents]
    customer_covariates, sale, profits, inventories, time_until_replenish = env.step(actions)
    inventory_history.append(inventories[0])
    newplot = env.render(True)
    if newplot:
        display.clear_output(wait=True)
        display.display(plt.gcf())
    if verbose:
        print('last customer covariate: ', last_customer_covariates)
        print('last (item bought, agent bought from, prices): ', sale)
        print('current_profit per agent: ', profits)
    last_customer_covariates = customer_covariates

plt.close()
print("Cumulative profit: {}".format(env.agent_profits))
print("Average revenue per customer: {}".format(np.asarray(env.agent_profits)/T))
print("Cumulative buyer utility: {}".format(env.cumulative_buyer_utility))

In [ ]:
plt.plot(inventory_history)